In [291]:
import numpy as np
import pandas as pd

In [292]:
data = {
    "study_hours": [2.0, 5.0, 3.0, 7.0],
    "sleep_hours": [6.0, 7.0, 5.5, 8.0],
    "passed": [0, 1, 0, 1],
}

df = pd.DataFrame(data)

print(df)

   study_hours  sleep_hours  passed
0          2.0          6.0       0
1          5.0          7.0       1
2          3.0          5.5       0
3          7.0          8.0       1


In [293]:
X = df[["study_hours", "sleep_hours"]].to_numpy()
y = df["passed"].to_numpy()

print(X)
print(y)

[[2.  6. ]
 [5.  7. ]
 [3.  5.5]
 [7.  8. ]]
[0 1 0 1]


**PREPARING THE DATA**

In [294]:
# transposing X
X = np.transpose(X)
print(X)

[[2.  5.  3.  7. ]
 [6.  7.  5.5 8. ]]


In [295]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def d_sigmoid(x):
    return sigmoid(x) * (1 - sigmoid(x))


A0 = X

**FEEDFORWARD PROPAGATION**

In [296]:
def initialise(s):
    np.random.seed(s)
    W1 = np.random.random((3, 2))
    b1 = np.random.random((3, 1))
    W2 = np.random.random((1, 3))
    b2 = np.random.random((1, 1))
    return W1, b1, W2, b2

In [297]:
def forward(A0, W1, W2, b1, b2):
    Z1 = W1 @ A0 + b1
    A1 = sigmoid(Z1)
    Z2 = W2 @ A1 + b2
    A2 = sigmoid(Z2)

    return Z1, A1, Z2, A2

**CALCULATING THE COST**

In [298]:
def cost(y, A0, A2):
    return np.sum((A2 - y) ** 2) / A0.shape[1]

**BACKPROPAGATION**

In [299]:
def dW1(y, A0, A2, Z1, Z2, W2):
    dW1 = 2 * (A2 - y)
    dW1 = dW1 * d_sigmoid(Z2)
    dW1 = (dW1.T @ W2).T
    dW1 = dW1 * d_sigmoid(Z1)
    dW1 = dW1 @ A0.T
    return dW1 / A0.shape[1]


def dW2(y, A0, A1, A2, Z2):
    dW2 = 2 * (A2 - y)
    dW2 = dW2 * d_sigmoid(Z2)
    dW2 = dW2 @ A1.T
    return dW2 / A0.shape[1]


def db1(y, A0, A2, Z1, Z2, W2):
    db1 = 2 * (A2 - y)
    db1 = db1 * d_sigmoid(Z2)
    db1 = (db1.T @ W2).T
    db1 = db1 * d_sigmoid(Z1)
    return np.sum(db1, axis=1, keepdims=True) / A0.shape[1]


def db2(y, A0, A2, Z2):
    db2 = 2 * (A2 - y)
    db2 = db2 * d_sigmoid(Z2)
    return np.sum(db2, axis=1, keepdims=True) / A0.shape[1]

In [300]:
def network(A0, y, s, n):
    W1_old, b1_old, W2_old, b2_old = initialise(s)
    Z1_old, A1_old, Z2_old, A2_old = forward(A0, W1_old, W2_old, b1_old, b2_old)
    old_cost = cost(y, A0, A2_old)

    W1_new = W1_old - n * dW1(y, A0, A2_old, Z1_old, Z2_old, W2_old)
    W2_new = W2_old - n * dW2(y, A0, A1_old, A2_old, Z2_old)
    b1_new = b1_old - n * db1(y, A0, A2_old, Z1_old, Z2_old, W2_old)
    b2_new = b2_old - n * db2(y, A0, A2_old, Z2_old)

    Z1_new, A1_new, Z2_new, A2_new = forward(A0, W1_new, W2_new, b1_new, b2_new)
    new_cost = cost(y, A0, A2_new)
    return old_cost, new_cost, Z1_new, A1_new, Z2_new, A2_new

In [301]:
def train_network_epochs(X, y, seed, epochs, learning_rate):
    W1, b1, W2, b2 = initialise(seed)

    for e in range(epochs):
        Z1, A1, Z2, A2 = forward(X, W1, W2, b1, b2)

        W1 = W1 - learning_rate * dW1(y, X, A2, Z1, Z2, W2)
        W2 = W2 - learning_rate * dW2(y, X, A1, A2, Z2)
        b1 = b1 - learning_rate * db1(y, X, A2, Z1, Z2, W2)
        b2 = b2 - learning_rate * db2(y, X, A2, Z2)

        Z1_new, A1_new, Z2_new, A2_new = forward(X, W1, W2, b1, b2)
        final_cost = cost(y, X, A2_new)

    return final_cost, W1, W2, b1, b2

In [302]:
def train_network_tolerance(X, y, seed, tolerance, learning_rate):
    W1, b1, W2, b2 = initialise(seed)

    Z1, A1, Z2, A2 = forward(X, W1, W2, b1, b2)
    old_cost = cost(y, A0, A2)

    while True:
        W1 = W1 - learning_rate * dW1(y, X, A2, Z1, Z2, W2)
        W2 = W2 - learning_rate * dW2(y, X, A1, A2, Z2)
        b1 = b1 - learning_rate * db1(y, X, A2, Z1, Z2, W2)
        b2 = b2 - learning_rate * db2(y, X, A2, Z2)

        Z1_new, A1_new, Z2_new, A2_new = forward(X, W1, W2, b1, b2)
        new_cost = cost(y, X, A2_new)

        if abs(new_cost - old_cost) <= tolerance:
            break

        old_cost = new_cost

    return old_cost, W1, W2, b1, b2

In [303]:
c1, c2, z1, a1, z2, a2 = network(A0, y, 42, 0.5)
print(c1, c2)

0.4234940420200924 0.41431242968705184


In [304]:
c_final, w1, w2, b1, b2 = train_network_epochs(A0, y, 42, 1000, 0.1)
print(c_final, "\n\n", w1, "\n\n", w2, "\n\n", b1, "\n\n", b2)

0.04578343321039619 

 [[ 0.37642869  0.95463535]
 [ 0.75610819  0.65608098]
 [ 2.41839583 -1.49659084]] 

 [[-0.41721004 -1.10292661  3.60529084]] 

 [[ 0.05883302]
 [ 0.87629233]
 [-0.17850611]] 

 [[-0.30522854]]


In [305]:
c_final, w1, w2, b1, b2 = train_network_tolerance(A0, y, 42, 0.000001, 0.1)
print(c_final, "\n\n", w1, "\n\n", w2, "\n\n", b1, "\n\n", b2)

0.4999597943540727 

 [[0.38744638 0.98020895]
 [0.76214671 0.67204908]
 [0.5622418  1.14362816]] 

 [[-2.57200208 -3.25718745 -1.86418522]] 

 [[0.0632665 ]
 [0.8788814 ]
 [0.77697302]] 

 [[-2.45318295]]
